# Notebook 10: Deep Collaborative Baseline — Neural Collaborative Filtering (NeuMF)

## 1. Architecture
NeuMF fuses Generalized Matrix Factorization (GMF) and a Multi-Layer Perceptron (MLP):
$$\phi^{\text{GMF}} = \mathbf{p}_u^G \odot \mathbf{q}_i^G, \qquad \phi^{\text{MLP}} = \text{MLP}([\mathbf{p}_u^M \| \mathbf{q}_i^M]), \qquad \hat{y}_{ui} = \mathbf{h}^\top [\phi^{\text{GMF}} \| \phi^{\text{MLP}}]$$

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Resolve repository root deterministically from ml/notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_DIR = os.path.join(REPO_ROOT, "ml", "data", "processed")
ARTIFACTS_DIR = os.path.join(REPO_ROOT, "ml", "artifacts")
VIZ_DIR = os.path.join(REPO_ROOT, "ml", "visualizations")
plt.rcParams["figure.dpi"] = 110
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from ml.src.data.dataset import RecSysDataPipeline, BPRDataset
from ml.src.evaluation.metrics import hit_rate_at_k, ndcg_at_k, mrr

torch.manual_seed(SEED)

class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, dim=32):
        super().__init__()
        self.u_gmf = nn.Embedding(num_users, dim)
        self.i_gmf = nn.Embedding(num_items, dim)
        self.u_mlp = nn.Embedding(num_users, dim)
        self.i_mlp = nn.Embedding(num_items, dim)
        self.mlp = nn.Sequential(nn.Linear(dim * 2, 64), nn.ReLU(), nn.Linear(64, dim), nn.ReLU())
        self.head = nn.Linear(dim * 2, 1, bias=False)

    def score(self, u, i):
        gmf = self.u_gmf(u) * self.i_gmf(i)
        mlp = self.mlp(torch.cat([self.u_mlp(u), self.i_mlp(i)], dim=1))
        return self.head(torch.cat([gmf, mlp], dim=1)).squeeze(1)

pipeline = RecSysDataPipeline(
    articles_path=os.path.join(DATA_DIR, "articles.json"),
    interactions_path=os.path.join(DATA_DIR, "interactions.csv"),
    embeddings_path=os.path.join(DATA_DIR, "article_embeddings.npy"),
)
model = NeuMF(pipeline.num_users, pipeline.num_items, dim=32)
loader = DataLoader(BPRDataset(pipeline.get_bpr_triplets("train", num_negatives=1)), batch_size=64, shuffle=True)
opt = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

losses = []
for ep in range(1, 7):
    model.train()
    el = []
    for u, pos, neg, w in loader:
        opt.zero_grad()
        loss = (-F.logsigmoid(model.score(u, pos) - model.score(u, neg)) * w).mean()
        loss.backward()
        opt.step()
        el.append(loss.item())
    losses.append(float(np.mean(el)))
    print(f"Epoch {ep:02d}/06 | NeuMF Pairwise Loss = {losses[-1]:.4f}")

print(f"NeuMF Total Trainable Parameters: {sum(p.numel() for p in model.parameters()):,}")

Epoch 01/06 | NeuMF Pairwise Loss = 0.8677


Epoch 02/06 | NeuMF Pairwise Loss = 0.8115


Epoch 03/06 | NeuMF Pairwise Loss = 0.6620


Epoch 04/06 | NeuMF Pairwise Loss = 0.3671


Epoch 05/06 | NeuMF Pairwise Loss = 0.1296


Epoch 06/06 | NeuMF Pairwise Loss = 0.0372
NeuMF Total Trainable Parameters: 271,904
